# Matched run: reasoning teacher on the instruct teacher's problems
Goal: a like-for-like teacher comparison. Run the cells top to bottom.
Needs an A100 or L4 GPU (Runtime > Change runtime type). Secret name for the GitHub token: `GITHUB_TOKEN_DISTILL`.
The generation cell can take hours. If Colab disconnects, reconnect, re-run the setup cells, then re-run the generation cell: it continues where it stopped (the output file is on Drive).

## 1. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
TOKEN = userdata.get('GITHUB_TOKEN_DISTILL')
REPO = 'dedeepyakm/reasoning-trace-distillation'
if not os.path.exists('/content/repo'):
    subprocess.run(['git','clone',f'https://{TOKEN}@github.com/{REPO}.git','/content/repo'],check=True)
    subprocess.run(['git','-C','/content/repo','remote','set-url','origin',f'https://github.com/{REPO}.git'],check=True)
else:
    subprocess.run(['git','-C','/content/repo','pull',f'https://{TOKEN}@github.com/{REPO}.git','main'],check=True)
DRIVE_DATA = '/content/drive/MyDrive/reasoning-distillation/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
if not os.path.exists('/content/repo/data'):
    os.symlink(DRIVE_DATA, '/content/repo/data')
os.chdir('/content/repo')
print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)

In [ ]:
!pip -q install transformers accelerate datasets matplotlib bitsandbytes
!nvidia-smi

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
from common.run_logging import run_step
PY = 'python'
OUT = 'data/traces_reasoning_matched_k5.jsonl'

## 2. Quick test: 1 problem, 2 samples, short limit (about 5 minutes)
Check the printed token counts and that the file gets rows. Delete the test file afterwards.

In [ ]:
TEST_OUT = 'data/_test_matched.jsonl'
run_step('matched_test', [PY,'generation/teacher_trace_generation_reasoning_matched.py','--problems-from','data/cut_instruct_final.jsonl','--out',TEST_OUT,'--limit','1','--k','2','--max-new-tokens','1024'])
print(open(TEST_OUT).readline()[:600] if os.path.exists(TEST_OUT) else 'no test file written')

In [ ]:
if os.path.exists(TEST_OUT): os.remove(TEST_OUT)

## 3. Full matched run (50 problems x 5 samples, 16384 tokens)
Precision `4bit` matches the first reasoning run. Change to `bf16` for full precision (A100/L4).

In [ ]:
PRECISION = '4bit'
run_step('matched_generation', [PY,'generation/teacher_trace_generation_reasoning_matched.py','--problems-from','data/cut_instruct_final.jsonl','--out',OUT,'--k','5','--max-new-tokens','16384','--precision',PRECISION])

## 4. Truncation check (what fraction hit the limit or has no answer)

In [ ]:
import json
rows = [json.loads(l) for l in open(OUT)]
n = len(rows)
hit = sum(r['hit_token_limit'] for r in rows)
noans = sum(r['predicted_answer'] is None for r in rows)
print(f'traces: {n}  problems: {len({r["id"] for r in rows})}')
print(f'hit token limit: {hit} ({100*hit/n:.1f}%)   no extractable answer: {noans} ({100*noans/n:.1f}%)')
print('max tokens used:', max(r['n_generated_tokens'] for r in rows))

## 5. Cut analysis, problem-set check, figures

In [ ]:
run_step('analyze_matched', [PY,'analysis/analyze_traces_v3.py','--trace-file',OUT,'--out','data/cut_reasoning_matched.jsonl','--with-difficulty'])

In [ ]:
run_step('compare_matched', [PY,'diagnostics/compare_problem_sets.py','--a','data/cut_instruct_final.jsonl','--b','data/cut_reasoning_matched.jsonl'])

In [ ]:
run_step('figures_matched', [PY,'reporting/make_figures.py','--cut-file','data/cut_reasoning_matched.jsonl','--label','reasoning_matched','--outdir','data/figures'])
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/figures/reasoning_matched_*.png')):
    print(p); display(Image(p))

## 6. Score signal retention on the matched traces (GPU, slow)
Set `RUN_SCORING = True` to run.

In [ ]:
RUN_SCORING = False
if RUN_SCORING:
    run_step('score_matched', [PY,'scoring/score_cut_v2.py','--cut-file','data/cut_reasoning_matched.jsonl','--trace-file',OUT,'--out','data/score_reasoning_matched.json'])
    run_step('figures_matched_scored', [PY,'reporting/make_figures.py','--cut-file','data/cut_reasoning_matched.jsonl','--score-file','data/score_reasoning_matched.json','--label','reasoning_matched','--outdir','data/figures'])

## 7. Save the logs to GitHub (logs only, never the data)

In [ ]:
subprocess.run(['git','add','logs'])
print(subprocess.run(['git','status','--short'],capture_output=True,text=True).stdout)

In [ ]:
subprocess.run(['git','-c','user.name=Dedeepya Korukonda','-c','user.email=korukondadedeepya2002@gmail.com','commit','-m','Matched run logs'])
subprocess.run(['git','push',f'https://{TOKEN}@github.com/{REPO}.git','main'])